# Euclid I.13：从射线顺序到两直角

这是原创教学重构，不是原文翻译。普通角取值在零与半周之间；有限实跑不证明一般命题。下列格按标准库 compile/exec 顺序执行，不是 Jupyter kernel。

In [1]:
from fractions import Fraction as F
from geometry import build_certificate, verify_certificate, angle_partition
from cases import evaluate
b,c,d,a=(0,0),(1,0),(-2,0),(1,2)
cert=build_certificate(b,c,d,a)
print(cert["ray_order_case"])
print(cert["rotation_product"])


before_perpendicular
['-10', '0']


## 1. 相邻角的方向见证
BC、BD 必须是反向射线，BA 必须离开底线。两个复数式旋转对相乘得负实数；每个虚部为正排除了反身角和周数混淆。这里没有把向量角度假定为有理数。

In [2]:
print("first:",cert["first_rotation"],"second:",cert["second_rotation"])
print("verified:",verify_certificate(cert,b,c,d,a))


first: ['1', '2'] second: ['-2', '4']
verified: True


## 2. 垂线两侧的两种顺序
辅助垂线应选择 BA 所在半平面。斜射线位于垂线之前或之后，必须分别安排小角加法；垂直情形可直接由直角定义处理。

In [3]:
for point in ((1,2),(0,2),(-1,2),(1,-2)):
    q=build_certificate(b,c,d,point)
    print(point,q["orientation"],q["ray_order_case"])


(1, 2) 1 before_perpendicular
(0, 2) 1 right
(-1, 2) 1 after_perpendicular
(1, -2) -1 before_perpendicular


## 3. 独立的有理角度模型
这层以周为单位，预先采用角度加法和直角为四分之一周。不是从上面的有理坐标计算角度。它演示加法分拆，不是用 1/2−t 的定义重新“证明”几何定理。

In [4]:
for t in (F(1,8),F(1,4),F(3,8)):
    p=angle_partition(t)
    print(str(t),[str(x) for x in p["ordered_sectors"]],str(p["sum"]))


1/8 ['1/8', '1/8', '1/4'] 1/2
1/4 ['1/4', '0', '1/4'] 1/2
3/8 ['1/4', '1/8', '1/8'] 1/2


## 4. 拒绝篡改与不满足前提的输入
证书必须绑定独立给出的输入，不能仅检查一行声称结论。共线但同向也不够。

In [5]:
from copy import deepcopy
bad=deepcopy(cert);bad["rotation_product"]=["10","0"]
print("tampering accepted:",verify_certificate(bad,b,c,d,a))
try:
    build_certificate(b,c,(2,0),a)
except ValueError as exc:
    print(type(exc).__name__+": "+str(exc))


tampering accepted: False
ValueError: BC and BD must be opposite rays, not merely collinear


## 5. 有限回归集
平移、旋转、反射、不等长底线射线均被取样。一般恒等式和角度公理的论证见 WALKTHROUGH，不由样本数量保证。

In [6]:
result=evaluate()
for key in ("positive_certificates","negative_certificates","angle_partition_cases","ray_order_counts"):
    print(key,result[key])


positive_certificates 360
negative_certificates 16
angle_partition_cases 66
ray_order_counts {'after_perpendicular': 144, 'right': 72, 'before_perpendicular': 144}


## 复盘
问：只见乘积为负实数，为什么还不够？答：如果不限制每个普通角在 (0,π)，可以混入反身角及整周；正虚部提供这一步的方向见证。

问：程序核验了 Euclid 的全部公理吗？答：没有。向量空间、点序与角度解释是现代模型前提。